# 05 — New Features Lab (16 orthogonal candidates)

**Goal:** build 16 new candidate features that are orthogonal to the 15 already in
`src/features.py`, and visualize each with distribution / probability / heatmap /
correlation / p-value / confidence plots.  Scoring on the challenge is Sharpe =
mean(pnl)/std(pnl)·16 on *position-sized* PnL, so sizing-relevant statistics
(decile mean **and** std, bootstrap CIs, top-vs-bottom decile t-tests) matter —
not only directional correlation.

**Rules:** no edits to `src/` or `scripts/`; self-contained; one plot output dir
(`notebooks/plots/new_features/`).  No parquet/pickle writes.

In [1]:
import re, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 96
RNG = np.random.default_rng(42)

ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'data'
OUT = ROOT / 'notebooks' / 'plots' / 'new_features'
OUT.mkdir(parents=True, exist_ok=True)
print('root   :', ROOT)
print('data   :', DATA)
print('output :', OUT)

root   : /Users/kenji/Desktop/Projects/datathon-2026
data   : /Users/kenji/Desktop/Projects/datathon-2026/data
output : /Users/kenji/Desktop/Projects/datathon-2026/notebooks/plots/new_features


In [2]:
bars_seen    = pd.read_parquet(DATA / 'bars_seen_train.parquet')
bars_unseen  = pd.read_parquet(DATA / 'bars_unseen_train.parquet')
headlines    = pd.read_parquet(DATA / 'headlines_seen_train.parquet')

halfway_close = bars_seen.groupby('session')['close'].last()
end_close     = bars_unseen.groupby('session')['close'].last()
target        = (end_close / halfway_close - 1.0).rename('target_return')

print(f'sessions: {bars_seen.session.nunique()}  headlines: {len(headlines)}')
print(f'target: mean={target.mean():+.4%}  std={target.std():.4%}  %pos={(target>0).mean():.1%}')

sessions: 1000  headlines: 9740
target: mean=+0.3531%  std=2.0434%  %pos=57.0%


In [3]:
# ---------- Price-feature helpers ----------
def _pivot(bars, col):
    return bars.pivot(index='session', columns='bar_ix', values=col).sort_index()

def _sma(x, w):
    return pd.Series(x).rolling(w, min_periods=w).mean().to_numpy()

def _std(x, w):
    return pd.Series(x).rolling(w, min_periods=w).std(ddof=0).to_numpy()

def _ema(x, span):
    return pd.Series(x).ewm(span=span, adjust=False).mean().to_numpy()

def compute_price_features(bars):
    C = _pivot(bars, 'close').to_numpy(float)   # (N, 50)
    O = _pivot(bars, 'open').to_numpy(float)
    H = _pivot(bars, 'high').to_numpy(float)
    L = _pivot(bars, 'low').to_numpy(float)
    idx = _pivot(bars, 'close').index

    log_ret = np.zeros_like(C)
    log_ret[:, 1:] = np.log(C[:, 1:] / C[:, :-1])

    feats = {}

    # 1. late_half_return
    feats['late_half_return'] = C[:, 49] / C[:, 25] - 1.0

    # 2. momentum_5v20: mean log-ret(bars 45..49) - mean log-ret(bars 25..44)
    late  = log_ret[:, 45:50].mean(axis=1)
    early = log_ret[:, 25:45].mean(axis=1)
    feats['momentum_5v20'] = late - early

    # 3, 4. bollinger_pos_49, bb_width (20-bar)
    # For each session compute rolling on close (bars 30..49 window = 20 bars)
    win = C[:, 30:50]
    m = win.mean(axis=1)
    s = win.std(axis=1, ddof=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        feats['bollinger_pos_49'] = np.where(s > 0, (C[:, 49] - m) / (2 * s), 0.0)
        feats['bb_width']         = np.where(m > 0, (4 * s) / m, 0.0)

    # 5. macd_hist_49 at bar 49
    def _row_ema(a, span):
        alpha = 2.0 / (span + 1.0)
        out = np.empty_like(a)
        out[:, 0] = a[:, 0]
        for t in range(1, a.shape[1]):
            out[:, t] = alpha * a[:, t] + (1 - alpha) * out[:, t-1]
        return out
    ema12 = _row_ema(C, 12)
    ema26 = _row_ema(C, 26)
    macd_line = ema12 - ema26
    signal = _row_ema(macd_line, 9)
    feats['macd_hist_49'] = (macd_line - signal)[:, 49]

    # 6. donchian_pos (bars 30..49)
    lo = win.min(axis=1)
    hi = win.max(axis=1)
    rng = hi - lo
    feats['donchian_pos'] = np.where(rng > 0, (C[:, 49] - lo) / rng, 0.5)

    # 7. vol_ratio_late_vs_early
    v_early = log_ret[:, 1:25].std(axis=1, ddof=0)
    v_late  = log_ret[:, 25:50].std(axis=1, ddof=0)
    feats['vol_ratio_late_vs_early'] = np.where(v_early > 0, v_late / v_early, 1.0)

    # 8. body_to_range_avg
    body  = np.abs(C - O)
    rngHL = np.maximum(H - L, 1e-12)
    feats['body_to_range_avg'] = (body / rngHL).mean(axis=1)

    # 9. wick_imbalance_avg
    upper = H - np.maximum(O, C)
    lower = np.minimum(O, C) - L
    feats['wick_imbalance_avg'] = ((upper - lower) / rngHL).mean(axis=1)

    # 10. abs_return_ac_lag1 (lag-1 autocorr of |log_ret|)
    abs_r = np.abs(log_ret[:, 1:])
    def _ac1(row):
        if row.std() < 1e-12:
            return 0.0
        return np.corrcoef(row[:-1], row[1:])[0, 1]
    feats['abs_return_ac_lag1'] = np.array([_ac1(r) for r in abs_r])

    # 11. max_abs_bar_return
    feats['max_abs_bar_return'] = np.max(np.abs(log_ret), axis=1)

    return pd.DataFrame(feats, index=idx)


# ---------- Headline-feature helpers ----------
_WORD = re.compile(r"[A-Za-z][A-Za-z']+")
_CAP  = re.compile(r"\b([A-Z][a-z]{2,})\b")
_POS_WORDS = set('beats raises record strong breakthrough surges wins secures exceeds upgrades expands completes approves launches'.split())
_NEG_WORDS = set('misses warns weak recalls withdraws cuts falls declines lawsuit probe suspends downgrades bankruptcy delays'.split())

def _sent(text):
    toks = [w.lower() for w in _WORD.findall(text)]
    p = sum(1 for w in toks if w in _POS_WORDS)
    n = sum(1 for w in toks if w in _NEG_WORDS)
    if p + n == 0:
        return 0.0
    return (p - n) / (p + n)

def compute_headline_features(headlines, bars, sessions):
    h = headlines.copy()
    h['text_low'] = h['headline'].str.lower()
    h['sent']     = h['headline'].apply(_sent)
    h['ent']      = h['headline'].apply(lambda t: _CAP.findall(t)[0] if _CAP.findall(t) else '')

    by_s = h.groupby('session')

    # 12. miss_flag
    miss_flag = by_s['text_low'].apply(lambda s: int(s.str.contains('misses').any())).reindex(sessions, fill_value=0)

    # 13. contract_count
    contract_count = by_s['text_low'].apply(lambda s: int(s.str.contains('contract').sum())).reindex(sessions, fill_value=0)

    # 14. headline_bar_gini (concentration over 5-bar bins)
    def _gini(counts):
        c = np.asarray(counts, dtype=float)
        if c.sum() == 0:
            return 0.0
        c = np.sort(c)
        n = len(c)
        cum = np.cumsum(c)
        return (n + 1 - 2 * (cum.sum() / cum[-1])) / n
    def _bar_gini(group):
        bins = (group['bar_ix'] // 10).clip(0, 4)  # 5 bins of 10 bars
        counts = bins.value_counts().reindex(range(5), fill_value=0).to_numpy()
        return _gini(counts)
    gini = by_s.apply(_bar_gini).reindex(sessions, fill_value=0.0)

    # 15. sentiment_price_divergence:
    # per-bar sentiment (sum of sent across headlines in that bar) correlated with per-bar close return.
    close_w = bars.pivot(index='session', columns='bar_ix', values='close').sort_index()
    ret = close_w.pct_change(axis=1)
    bar_sent = h.groupby(['session', 'bar_ix'])['sent'].sum().unstack('bar_ix')
    bar_sent = bar_sent.reindex(index=sessions, columns=range(50), fill_value=0.0)
    def _row_corr(r_row, s_row):
        mask = (~np.isnan(r_row)) & (s_row != 0.0)
        if mask.sum() < 3:
            return 0.0
        r_sub = r_row[mask]; s_sub = s_row[mask]
        if np.std(r_sub) < 1e-12 or np.std(s_sub) < 1e-12:
            return 0.0
        rho, _ = stats.spearmanr(r_sub, s_sub)
        return rho if np.isfinite(rho) else 0.0
    divergence = []
    for sess in sessions:
        rvec = ret.loc[sess].to_numpy()
        svec = bar_sent.loc[sess].to_numpy()
        divergence.append(_row_corr(rvec, svec))
    divergence = pd.Series(divergence, index=sessions, name='sentiment_price_divergence')

    # 16. entity_dominance
    def _ent_dom(group):
        ents = [e for e in group['ent'] if e]
        if not ents:
            return 0.0
        vc = pd.Series(ents).value_counts()
        return float(vc.iloc[0] / vc.sum())
    ent_dom = by_s.apply(_ent_dom).reindex(sessions, fill_value=0.0)

    return pd.DataFrame({
        'miss_flag':                  miss_flag.astype(float),
        'contract_count':             contract_count.astype(float),
        'headline_bar_gini':          gini.astype(float),
        'sentiment_price_divergence': divergence.astype(float),
        'entity_dominance':           ent_dom.astype(float),
    })


def compute_features(bars, headlines):
    price = compute_price_features(bars)
    nlp   = compute_headline_features(headlines, bars, price.index)
    return pd.concat([price, nlp], axis=1)

In [4]:
feat = compute_features(bars_seen, headlines)
df   = feat.join(target)
FEATURE_NAMES = list(feat.columns)
print(f'{len(FEATURE_NAMES)} features, {len(df)} sessions')
df.describe().T.round(4)

16 features, 1000 sessions


,count,mean,std,min,25%,50%,75%,max
late_half_return,1000.0,0.0009,0.0150,-0.0623,-0.0086,0.0006,0.0102,0.0576
momentum_5v20,1000.0,-0.0000,0.0015,-0.0049,-0.0009,0.0000,0.0009,0.0053
bollinger_pos_49,1000.0,0.0133,0.6716,-1.4111,-0.5529,0.0482,0.5792,1.4345
bb_width,1000.0,0.0196,0.0093,0.0049,0.0131,0.0174,0.0240,0.0639
macd_hist_49,1000.0,-0.0000,0.0010,-0.0034,-0.0007,0.0000,0.0006,0.0036
donchian_pos,1000.0,0.5094,0.3673,0.0000,0.1478,0.5318,0.8723,1.0000
vol_ratio_late_vs_early,1000.0,1.0158,0.2202,0.4471,0.8571,0.9961,1.1605,1.8652
body_to_range_avg,1000.0,0.5015,0.0388,0.4041,0.4766,0.5006,0.5278,0.6204
wick_imbalance_avg,1000.0,-0.0031,0.0492,-0.1666,-0.0368,-0.0037,0.0290,0.1546
abs_return_ac_lag1,1000.0,-0.0225,0.1359,-0.5171,-0.1168,-0.0256,0.0651,0.4384


In [5]:
# Leakage check: prove the 16 features depend only on seen-half data (bars 0-49).
# If compute_features() ran on public_test data — which has NO bars 50-99 — still
# produces finite values for all 16 columns, it cannot be using the unseen half.
bars_pub    = pd.read_parquet(DATA / 'bars_seen_public_test.parquet')
head_pub    = pd.read_parquet(DATA / 'headlines_seen_public_test.parquet')
assert bars_pub['bar_ix'].max() == 49, 'public_test must only contain bars 0-49'
feat_pub = compute_features(bars_pub, head_pub)
print(f'public_test: {len(feat_pub)} sessions, 16 features')
print(f'  NaN fraction per column: {feat_pub.isna().mean().max():.4f} (max across cols)')
assert feat_pub.shape[1] == 16, 'feature column count changed'
assert feat_pub.isna().mean().max() < 0.01, 'too many NaNs — suspect a latent dependency'
print('PASS — features are computable from seen-half data only. No leakage.')

public_test: 10000 sessions, 16 features
  NaN fraction per column: 0.0000 (max across cols)
PASS — features are computable from seen-half data only. No leakage.


In [6]:
def _bootstrap_r(x, y, n=1000):
    n_samples = len(x)
    idx = RNG.integers(0, n_samples, size=(n, n_samples))
    rs = np.array([stats.pearsonr(x[i], y[i])[0] for i in idx])
    return np.percentile(rs, [2.5, 97.5])

def plot_feature(name, x, y, out_path):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    r_p,  p_p = stats.pearsonr(x, y)
    r_s,  p_s = stats.spearmanr(x, y)
    ci_lo, ci_hi = _bootstrap_r(x, y, n=1000)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(f'{name}', fontsize=14, fontweight='bold')

    # (0,0) histogram + KDE
    ax = axes[0, 0]
    sns.histplot(x, kde=True, ax=ax, color='steelblue', stat='density', bins=40)
    ax.axvline(np.mean(x), color='red', ls='--', lw=1, label=f'mean={np.mean(x):.4g}')
    ax.set_title('Distribution'); ax.set_xlabel(name); ax.legend(fontsize=8)

    # (0,1) Q-Q plot
    ax = axes[0, 1]
    stats.probplot(x, dist='norm', plot=ax)
    ax.set_title('Q-Q plot (vs normal)')

    # (0,2) scatter vs target with OLS + 95% CI
    ax = axes[0, 2]
    sns.regplot(x=x, y=y, ax=ax, ci=95, scatter_kws={'alpha':0.25, 's':10}, line_kws={'color':'red'})
    ax.set_title(f'vs target_return  (r={r_p:+.3f}, p={p_p:.1e})')
    ax.set_xlabel(name); ax.set_ylabel('target_return')

    # (1,0) decile mean target with bootstrap CI
    ax = axes[1, 0]
    try:
        q = pd.qcut(x, 10, labels=False, duplicates='drop')
        q = np.asarray(q, dtype=float)
    except Exception:
        q = np.full_like(x, np.nan, dtype=float)
    valid = ~np.isnan(q)
    unique_q = sorted(np.unique(q[valid]).astype(int).tolist())
    means, los, his = [], [], []
    if len(unique_q) >= 2:
        for qi in unique_q:
            yq = y[(q == qi) & valid]
            if len(yq) == 0:
                means.append(np.nan); los.append(np.nan); his.append(np.nan); continue
            bs = [yq[RNG.integers(0, len(yq), len(yq))].mean() for _ in range(500)]
            means.append(yq.mean())
            los.append(np.percentile(bs, 2.5))
            his.append(np.percentile(bs, 97.5))
        means = np.array(means); los = np.array(los); his = np.array(his)
        ax.bar(unique_q, means, yerr=[means-los, his-means], color='steelblue',
               edgecolor='black', capsize=3, alpha=0.8)
        ax.axhline(0, color='k', lw=0.8)
        ax.set_title('Decile mean target (95% bootstrap CI)')
        ax.set_xlabel('decile'); ax.set_ylabel('mean target_return')
        top = y[q == unique_q[-1]]; bot = y[q == unique_q[0]]
        if len(top) > 1 and len(bot) > 1:
            t_stat, t_p = stats.ttest_ind(top, bot, equal_var=False)
            dmu = top.mean() - bot.mean()
        else:
            t_stat, t_p, dmu = np.nan, np.nan, np.nan
    else:
        # fallback: binary split by x median (for low-variance features)
        med = np.median(x)
        top = y[x > med]; bot = y[x <= med]
        if len(top) > 1 and len(bot) > 1:
            t_stat, t_p = stats.ttest_ind(top, bot, equal_var=False)
            dmu = top.mean() - bot.mean()
        else:
            t_stat, t_p, dmu = np.nan, np.nan, np.nan
        ax.text(0.5, 0.5, 'not enough unique values\nfor deciles',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Decile plot (skipped)')

    # (1,1) 2D heatmap: feature-quintile x target-quintile joint density
    ax = axes[1, 1]
    try:
        fx = pd.qcut(x, 5, labels=False, duplicates='drop')
        fy = pd.qcut(y, 5, labels=False, duplicates='drop')
        joint = pd.crosstab(fx, fy, normalize=True)
        if joint.size > 1:
            sns.heatmap(joint, ax=ax, cmap='RdBu_r', center=0.04, annot=True, fmt='.2f',
                        cbar_kws={'label':'P(feat_q, target_q)'})
            ax.set_xlabel('target quintile'); ax.set_ylabel('feature quintile')
        else:
            ax.text(0.5, 0.5, 'degenerate joint dist', ha='center', va='center', transform=ax.transAxes)
    except Exception as e:
        ax.text(0.5, 0.5, f'heatmap failed\n{e}', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Joint density (quintiles)')

    # (1,2) stats annotation panel
    ax = axes[1, 2]; ax.axis('off')
    bonf = 0.05 / 16
    sig_p = 'YES' if p_p < bonf else 'no'
    sig_s = 'YES' if p_s < bonf else 'no'
    tp_str = f'{t_p:.2e}' if np.isfinite(t_p) else 'n/a'
    ts_str = f'{t_stat:+.3f}' if np.isfinite(t_stat) else 'n/a'
    dmu_str = f'{dmu:+.4%}' if np.isfinite(dmu) else 'n/a'
    txt = (
        f'N           : {len(x):>6d}\n'
        f'mean / std  : {np.mean(x):+.4g} / {np.std(x):.4g}\n'
        f'skew / kurt : {stats.skew(x):+.3f} / {stats.kurtosis(x):+.3f}\n'
        f'\n'
        f'Pearson r   : {r_p:+.4f}  (p={p_p:.2e})  sig(Bonf)={sig_p}\n'
        f'Spearman rho: {r_s:+.4f}  (p={p_s:.2e})  sig(Bonf)={sig_s}\n'
        f'bootstrap CI: [{ci_lo:+.3f}, {ci_hi:+.3f}]\n'
        f'\n'
        f'top-vs-bot  : delta_mu={dmu_str}\n'
        f't-stat      : {ts_str}  (p={tp_str})\n'
        f'\n'
        f'Bonferroni threshold (16 tests): alpha={bonf:.4f}'
    )
    ax.text(0.02, 0.98, txt, family='monospace', va='top', fontsize=10)
    ax.set_title('Summary statistics')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, dpi=110, bbox_inches='tight')
    plt.close(fig)

    return {'feature': name, 'pearson_r': r_p, 'pearson_p': p_p,
            'spearman_rho': r_s, 'spearman_p': p_s,
            'ci_lo': ci_lo, 'ci_hi': ci_hi,
            'top_bot_delta': dmu, 't_p': t_p}

In [7]:
results = []
y = df['target_return'].to_numpy()
for i, name in enumerate(FEATURE_NAMES, 1):
    x = df[name].to_numpy()
    out_path = OUT / f'feat_{i:02d}_{name}.png'
    results.append(plot_feature(name, x, y, out_path))
    print(f'  [{i:2d}/{len(FEATURE_NAMES)}] {name:32s} -> {out_path.name}')

results_df = pd.DataFrame(results).set_index('feature')
results_df.round(4)

  [ 1/16] late_half_return                 -> feat_01_late_half_return.png


  [ 2/16] momentum_5v20                    -> feat_02_momentum_5v20.png


  [ 3/16] bollinger_pos_49                 -> feat_03_bollinger_pos_49.png


  [ 4/16] bb_width                         -> feat_04_bb_width.png


  [ 5/16] macd_hist_49                     -> feat_05_macd_hist_49.png


...

  [13/16] contract_count                   -> feat_13_contract_count.png


  [14/16] headline_bar_gini                -> feat_14_headline_bar_gini.png


  [15/16] sentiment_price_divergence       -> feat_15_sentiment_price_divergence.png


  [16/16] entity_dominance                 -> feat_16_entity_dominance.png


,pearson_r,pearson_p,spearman_rho,spearman_p,ci_lo,ci_hi,top_bot_delta,t_p
feature,,,,,,,,
late_half_return,-0.0686,0.0300,-0.0693,0.0285,-0.1359,-0.0057,-0.0033,0.2762
momentum_5v20,0.0654,0.0387,0.0721,0.0226,-0.0051,0.1346,0.0053,0.0903
bollinger_pos_49,-0.0358,0.2577,-0.0344,0.2770,-0.0953,0.0262,0.0008,0.7875
bb_width,0.0222,0.4834,0.0229,0.4698,-0.0461,0.0876,-0.0020,0.4955
macd_hist_49,-0.0054,0.8646,-0.0155,0.6244,-0.0715,0.0653,0.0015,0.6188
donchian_pos,-0.0487,0.1241,-0.0462,0.1444,-0.1124,0.0115,-0.0004,0.8447
vol_ratio_late_vs_early,-0.0309,0.3282,-0.0436,0.1681,-0.0913,0.0324,-0.0037,0.1711
body_to_range_avg,0.0200,0.5280,0.0259,0.4141,-0.0388,0.0777,-0.0005,0.8525
wick_imbalance_avg,0.0200,0.5276,0.0310,0.3270,-0.0388,0.0769,0.0031,0.2984


In [8]:
# Recompute production-15 feature sketches INLINE (no src/ imports)
# Reasonable approximations for cross-redundancy check.
Cw = bars_seen.pivot(index='session', columns='bar_ix', values='close').sort_index()
Ow = bars_seen.pivot(index='session', columns='bar_ix', values='open').sort_index()
Hw = bars_seen.pivot(index='session', columns='bar_ix', values='high').sort_index()
Lw = bars_seen.pivot(index='session', columns='bar_ix', values='low').sort_index()
C = Cw.to_numpy(); O = Ow.to_numpy(); H = Hw.to_numpy(); L = Lw.to_numpy()
log_ret = np.zeros_like(C); log_ret[:,1:] = np.log(C[:,1:]/C[:,:-1])

prod = pd.DataFrame(index=Cw.index)
prod['fh_return']       = C[:,49]/C[:,0] - 1.0
prod['recent_return_3'] = C[:,49]/C[:,46] - 1.0
rng_bar = np.maximum(H[:,49]-L[:,49], 1e-12)
prod['upper_wick_49']   = (H[:,49] - np.maximum(O[:,49], C[:,49])) / rng_bar
running_max = np.maximum.accumulate(C, axis=1)
prod['max_drawdown_fh'] = (C/running_max - 1.0).min(axis=1)
prod['parkinson_vol']   = np.sqrt((np.log(H/np.maximum(L,1e-12))**2).mean(axis=1) / (4*np.log(2)))
with np.errstate(invalid='ignore', divide='ignore'):
    gk = 0.5*np.log(H/np.maximum(L,1e-12))**2 - (2*np.log(2)-1)*np.log(C/np.maximum(O,1e-12))**2
prod['garman_klass_vol'] = np.sqrt(np.nanmean(gk, axis=1))
with np.errstate(invalid='ignore', divide='ignore'):
    rs = np.log(H/np.maximum(C,1e-12))*np.log(H/np.maximum(O,1e-12)) + np.log(L/np.maximum(C,1e-12))*np.log(L/np.maximum(O,1e-12))
prod['rogers_satchell_vol'] = np.sqrt(np.nanmean(rs, axis=1))
prod['yz_vol_approx']       = log_ret[:,1:].std(axis=1, ddof=0)
prod['realized_skewness']   = stats.skew(log_ret[:,1:], axis=1, bias=False)
# RSI-14 at bar 49
delta = np.diff(C, axis=1)
up   = np.where(delta > 0, delta, 0.0)
down = np.where(delta < 0, -delta, 0.0)
def _rsi(u, d, span=14):
    u_ema = _row_ema_func(u, span); d_ema = _row_ema_func(d, span)
    with np.errstate(invalid='ignore', divide='ignore'):
        rs = u_ema / np.maximum(d_ema, 1e-12)
        return 100.0 - 100.0/(1.0 + rs)
def _row_ema_func(a, span):
    alpha = 1.0/span
    out = np.empty_like(a, dtype=float); out[:,0] = a[:,0]
    for t in range(1, a.shape[1]):
        out[:,t] = alpha*a[:,t] + (1-alpha)*out[:,t-1]
    return out
prod['rsi_14'] = _rsi(up, down)[:,-1]

# headline-based prod sketches
h_low = headlines.copy(); h_low['t']=h_low['headline'].str.lower()
_bull = re.compile(r'beat|raise|record|surge|secure|exceed|breakthrough|upgrade|expand')
_bear = re.compile(r'miss|warn|weak|recall|withdraw|cut|decline|lawsuit|probe|suspend|downgrade|bankruptcy')
bull = h_low.groupby('session')['t'].apply(lambda s: int(s.str.contains(_bull).sum())).reindex(Cw.index, fill_value=0)
bear = h_low.groupby('session')['t'].apply(lambda s: int(s.str.contains(_bear).sum())).reindex(Cw.index, fill_value=0)
prod['bmb'] = (bull - bear).astype(float)
# sentiment approximations
sent_bar = headlines.assign(s=headlines['headline'].apply(_sent)).groupby(['session','bar_ix'])['s'].sum().unstack('bar_ix')
sent_bar = sent_bar.reindex(index=Cw.index, columns=range(50), fill_value=0.0)
prod['sent_sma10']   = sent_bar.iloc[:, 40:50].mean(axis=1)
def _row_ewm_last(arr, span):
    alpha = 2.0 / (span + 1.0)
    a = np.asarray(arr, dtype=float)
    out = a[:, 0].copy()
    for t in range(1, a.shape[1]):
        out = alpha * a[:, t] + (1 - alpha) * out
    return out
prod['sent_ema20']   = _row_ewm_last(sent_bar.to_numpy(), 20)
w = np.exp(-(49 - np.arange(50))/20.0)
prod['sent_wt_decay20'] = (sent_bar.to_numpy() * w).sum(axis=1)

big = pd.concat([feat, prod, target], axis=1)
corr = big.corr(method='spearman')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=False, square=True, cbar_kws={'label': 'Spearman ρ'})
ax.set_title('Spearman correlation: 16 new (top-left) + 15 production sketches + target', fontsize=12)
plt.tight_layout()
fig.savefig(OUT / '00_corr_heatmap.png', dpi=110, bbox_inches='tight')
plt.close(fig)

# redundancy check: any new feature with |ρ| > 0.7 to any production one?
new_vs_prod = corr.loc[FEATURE_NAMES, prod.columns]
dup = new_vs_prod[new_vs_prod.abs() > 0.7].stack()
print('Redundant pairs (|ρ|>0.7):')
print(dup if len(dup) else '  (none — all 16 new features are orthogonal)')

Redundant pairs (|ρ|>0.7):
late_half_return  fh_return         NaN
                  recent_return_3   NaN
                  upper_wick_49     NaN
                  max_drawdown_fh   NaN
                  parkinson_vol     NaN
                                     ..
entity_dominance  rsi_14            NaN
                  bmb               NaN
                  sent_sma10        NaN
                  sent_ema20        NaN
                  sent_wt_decay20   NaN
Length: 224, dtype: float64


In [9]:
r_df = results_df[['pearson_r', 'pearson_p', 'spearman_rho', 'spearman_p']].copy()
r_df['abs_pearson'] = r_df['pearson_r'].abs()
r_df = r_df.sort_values('abs_pearson', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['tomato' if p < 0.05/16 else 'steelblue' for p in r_df['pearson_p']]
ax.barh(r_df.index, r_df['pearson_r'], color=colors, edgecolor='black')
ax.axvline(0, color='k', lw=0.8)
ax.axvline(0.05,  color='grey', ls='--', lw=0.6, alpha=0.7)
ax.axvline(-0.05, color='grey', ls='--', lw=0.6, alpha=0.7)
ax.set_xlabel('Pearson r with target_return')
ax.set_title('Pearson r per feature (red = Bonferroni-significant at α=0.05/16)')
plt.tight_layout()
fig.savefig(OUT / '00_abs_r_bars.png', dpi=110, bbox_inches='tight')
plt.close(fig)
r_df[['pearson_r','pearson_p','spearman_rho','spearman_p']].round(4)

,pearson_r,pearson_p,spearman_rho,spearman_p
feature,,,,
headline_bar_gini,0.0041,0.8978,0.0112,0.7239
macd_hist_49,-0.0054,0.8646,-0.0155,0.6244
abs_return_ac_lag1,-0.0104,0.7426,-0.0088,0.7803
contract_count,-0.0154,0.6262,-0.0067,0.8329
body_to_range_avg,0.0200,0.5280,0.0259,0.4141
wick_imbalance_avg,0.0200,0.5276,0.0310,0.3270
bb_width,0.0222,0.4834,0.0229,0.4698
miss_flag,-0.0236,0.4566,-0.0115,0.7174
entity_dominance,0.0286,0.3671,0.0381,0.2291


In [10]:
fig, ax = plt.subplots(figsize=(9, 7))
abs_r = results_df['pearson_r'].abs()
neglog_p = -np.log10(results_df['pearson_p'].clip(lower=1e-300))
colors = ['tomato' if p < 0.05/16 else 'steelblue' for p in results_df['pearson_p']]
ax.scatter(abs_r, neglog_p, c=colors, s=60, edgecolor='black')
for name in results_df.index:
    ax.annotate(name, (abs_r[name], neglog_p[name]), fontsize=8,
                xytext=(4, 4), textcoords='offset points')
ax.axhline(-np.log10(0.05/16), color='red', ls='--', label=f'Bonferroni α=0.05/16')
ax.axhline(-np.log10(0.05),    color='grey', ls='--', label='α=0.05')
ax.set_xlabel('|Pearson r|'); ax.set_ylabel('-log10(p-value)')
ax.set_title('Feature funnel plot (effect size vs significance)')
ax.legend()
plt.tight_layout()
fig.savefig(OUT / '00_funnel.png', dpi=110, bbox_inches='tight')
plt.close(fig)

In [11]:
try:
    import lightgbm as lgb
    from sklearn.model_selection import KFold

    X = feat.to_numpy(); y = target.loc[feat.index].to_numpy()
    imp = np.zeros(X.shape[1])
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for tr, va in kf.split(X):
        m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.02,
                              num_leaves=15, min_data_in_leaf=30,
                              feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
                              verbose=-1, random_state=42)
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], callbacks=[lgb.early_stopping(30, verbose=False)])
        imp += m.booster_.feature_importance(importance_type='gain')
    imp /= 5.0
    imp_df = pd.Series(imp, index=FEATURE_NAMES).sort_values()

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_df.index, imp_df.values, color='darkorange', edgecolor='black')
    ax.set_xlabel('mean gain importance (5-fold CV)')
    ax.set_title('LightGBM feature importance (16 new features → target_return)')
    plt.tight_layout()
    fig.savefig(OUT / '00_lgbm_importance.png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(imp_df.sort_values(ascending=False).round(1).to_string())
except Exception as e:
    print(f'LightGBM step skipped: {e}')

abs_return_ac_lag1            0.0
headline_bar_gini             0.0
body_to_range_avg             0.0
vol_ratio_late_vs_early       0.0
momentum_5v20                 0.0
max_abs_bar_return            0.0
late_half_return              0.0
bb_width                      0.0
entity_dominance              0.0
donchian_pos                  0.0
bollinger_pos_49              0.0
macd_hist_49                  0.0
wick_imbalance_avg            0.0
contract_count                0.0
miss_flag                     0.0
sentiment_price_divergence    0.0


In [12]:
rank = results_df.copy()
rank['abs_r'] = rank['pearson_r'].abs()
rank = rank.sort_values('abs_r', ascending=False)
print('Features ranked by |Pearson r| with target_return\n')
print(rank[['pearson_r', 'pearson_p', 'spearman_rho', 'spearman_p',
            'ci_lo', 'ci_hi', 'top_bot_delta', 't_p']].round(4).to_string())

Features ranked by |Pearson r| with target_return

                            pearson_r  pearson_p  spearman_rho  spearman_p   ci_lo   ci_hi  top_bot_delta     t_p
feature                                                                                                          
late_half_return              -0.0686     0.0300       -0.0693      0.0285 -0.1359 -0.0057        -0.0033  0.2762
max_abs_bar_return             0.0668     0.0348        0.0886      0.0051  0.0071  0.1316         0.0052  0.0598
momentum_5v20                  0.0654     0.0387        0.0721      0.0226 -0.0051  0.1346         0.0053  0.0903
donchian_pos                  -0.0487     0.1241       -0.0462      0.1444 -0.1124  0.0115        -0.0004  0.8447
bollinger_pos_49              -0.0358     0.2577       -0.0344      0.2770 -0.0953  0.0262         0.0008  0.7875
vol_ratio_late_vs_early       -0.0309     0.3282       -0.0436      0.1681 -0.0913  0.0324        -0.0037  0.1711
entity_dominance               0.0286

## Ranking

Features above are sorted by |Pearson r| with `target_return`.  Red bars / red
points in `00_abs_r_bars.png` and `00_funnel.png` are **Bonferroni-significant**
at α = 0.05/16 ≈ 0.003.  A feature only deserves promotion into
`src/features.py` if it is **both** orthogonal to the existing 15 (check
`00_corr_heatmap.png`) **and** has a stable, statistically significant
directional signal.

The top 3–5 in this ranking are the candidates to add in a follow-up task.

### Outputs written to `notebooks/plots/new_features/`
- `feat_01_…` … `feat_16_…`  per-feature diagnostic grid
- `00_corr_heatmap.png`        cross-feature Spearman heatmap (new × prod × target)
- `00_abs_r_bars.png`          Pearson r per feature with Bonferroni coloring
- `00_funnel.png`              |r| vs -log10(p) funnel
- `00_lgbm_importance.png`     LightGBM gain importance (optional)